In [16]:
import pymupdf4llm
import sys
import os
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from controllers.utils import extract_text, clean_markdown

In [17]:
path = os.path.join(os.getcwd(), "../docs/simCLR.pdf")
print(path)

/Users/ayushjaswal/Developer/personal-projects/groundedQAproject/notebooks/../docs/simCLR.pdf


In [18]:
mkdn = clean_markdown(extract_text(path))

In [ ]:
# from IPython.display import display, Markdown

# display(Markdown(mkdn))

In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter, Language

header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[
        ("#", "title"),
        ("##", "section"),
        ("###", "subsection"),
    ]
)

header_chunks = header_splitter.split_text(mkdn)

char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

final_chunks = char_splitter.split_documents(header_chunks)
print(f"Total chunks: {len(final_chunks)}")
print("\n--- Chunk 0 ---")
print(final_chunks[0].page_content)
print(final_chunks[0].metadata)

/Users/ayushjaswal/Developer/personal-projects/groundedQAproject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total chunks: 106

--- Chunk 0 ---
**A Simple Framework for Contrastive Learning of Visual Representations**  
**Ting Chen**[1] **Simon Kornblith**[1] **Mohammad Norouzi**[1] **Geoffrey Hinton**[1]
{}


In [ ]:
len(final_chunks)

106

In [ ]:
from sentence_transformers import SentenceTransformer

embeddings_model = SentenceTransformer("all-MiniLM-L6-v2")
print(embeddings_model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4375.17it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [ ]:
texts = [chunk.page_content for chunk in final_chunks]

embeddings = embeddings_model.encode(texts, show_progress_bar=True)


print(f"Chunks: {len(texts)}")
print(f"Embedding shape: {embeddings.shape}") 

Batches: 100%|██████████| 4/4 [00:00<00:00,  4.29it/s]

Chunks: 106
Embedding shape: (106, 384)


In [ ]:
embeddings_list = embeddings.tolist()
type(embeddings_list), type(embeddings_list[0])

(list, list)

In [8]:
import chromadb

chroma_client = chromadb.PersistentClient(path="../knowledgebase")

# metadatas = []
# for i, chunk in enumerate(final_chunks):
#     meta = chunk.metadata.copy() 
#     meta["chunk_id"] = i
#     meta["source"] = "simCLR.pdf"
#     metadatas.append(meta)

# reindex
# chroma_client.delete_collection(name="knowledgebase")
collection = chroma_client.get_or_create_collection(name="knowledgebase")

# collection.add(
#     embeddings=embeddings.tolist(),
#     documents=texts,
#     ids=[str(i) for i in range(len(texts))],
#     metadatas=metadatas
# )

# print(f"Stored {collection.count()} chunks")

In [ ]:
#print(collection.get('160', include=['embeddings', 'documents', 'metadatas']))

In [ ]:
query = "what is sim?"

results = collection.query(
    query_texts=[query],
    n_results=4,
    include=["documents", "distances", "metadatas"]
)

In [ ]:
results

{'ids': [['8', '100', '7', '5']],
 'embeddings': None,
 'documents': [['Inspired by recent contrastive learning algorithms (see Section 7 for an overview), SimCLR learns representations by maximizing agreement between differently augmented views of the same data example via a contrastive loss in the latent space. As illustrated in Figure 2, this framework comprises the following four major components.  \n- A stochastic _data augmentation_ module that transforms any given data example randomly resulting in two correlated views of the same example, denoted _**x**_ ˜ _i_ and _**x**_ ˜ _j_ , which we consider as a positive pair. In this work, we sequentially apply three simple augmentations: _random cropping_ followed by resize back to the original size, _random color distortions_ , and _random Gaussian blur_ . As shown in Section 3, the combination of random crop and color distortion is crucial to achieve a good performance.',
   'As we have noted in the main text, most individual compone

In [ ]:
from IPython.display import display, Markdown
display(Markdown(collection.get("7")['documents'][0]))

We combine these findings to achieve a new state-of-the-art in self-supervised and semi-supervised learning on ImageNet ILSVRC-2012 (Russakovsky et al., 2015). Under the linear evaluation protocol, SimCLR achieves 76.5% top-1 accuracy, which is a 7% relative improvement over previous state-of-the-art (Hénaff et al., 2019). When fine-tuned with only 1% of the ImageNet labels, SimCLR achieves 85.8% top-5 accuracy, a relative improvement of 10% (Hénaff et al., 2019). When fine-tuned on other natural image classification datasets, SimCLR performs on par with or better than a strong supervised baseline (Kornblith et al., 2019) on 10 out of 12 datasets.

In [ ]:
# queries = [
#     "what is SimCLR?",
#     "what data augmentation does SimCLR use?",
#     "how does the contrastive loss work?",
#     "what is the projection head?",
# ]

queries = [
    "SimCLR is a framework for contrastive learning",        # rephrased
    "what data augmentation does SimCLR use?",               # already good
    "how does the contrastive loss work?",                   # already good  
    "projection head architecture non-linear MLP",           # rephrased
]

def scoreboard(distances):
    score = 0
    weights = [1.0, 0.75, 0.55, 0,35]
    for val, weights in zip(distances[0], weights):
        if val > 1.35:
            score += -1 * weights
        elif val < 1:
            score += 1 * weights
        else:
            score += 0.5 * weights
    return round(score, 2)
        

# scores = []
for query in queries:
    results = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query}")
    print(f"Score: {scoreboard(results['distances'])}")

Query: SimCLR is a framework for contrastive learning
Score: 2.3
Query: what data augmentation does SimCLR use?
Score: 2.02
Query: how does the contrastive loss work?
Score: 2.3
Query: projection head architecture non-linear MLP
Score: 1.65


In [5]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv(override=True)

login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
from huggingface_hub import InferenceClient

hf_client = InferenceClient()

def hyde_query(question):
    response = hf_client.chat_completion(
        model="meta-llama/Llama-3.1-8B-Instruct",
        messages=[
            {
                "role": "system",
                "content": "You are a engineering tutor assistant. Given a question, write a short factual paragraph that directly answers it, as if you were writing it in a research paper. Do not ask questions back."
            },
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=150,
    )
    return response.choices[0].message.content

# test it
hyde_q = hyde_query("what is SimCLR in context of Machien Learning?")
print(hyde_q)

SimCLR (Simple Framework for Contrastive Learning of Visual Representations) is a popular unsupervised learning technique in the context of Machine Learning, specifically for Computer Vision tasks. Introduced in 2020, SimCLR enhances the original concept of contrastive learning by using a data augmentation pipeline to generate augmented views of the same image and then trains a neural network to predict their similarity. The architecture of SimCLR consists of an encoder network (e.g., a deep neural network) that is trained using a contrastive loss function to learn robust and generalizable representations. The contrastive loss measures the similarity between the encoder's outputs for two different augmented views of the same image, thereby encouraging the network to learn a common and invariant representation across varying transformations. The


In [7]:
# queries = [
#     "what is SimCLR?",
#     "what data augmentation does SimCLR use?",
#     "how does the contrastive loss work?",
#     "what is the projection head?",
# ]

queries = [
    "SimCLR is a framework for contrastive learning",        # rephrased
    "what data augmentation does SimCLR use?",               # already good
    "how does the contrastive loss work?",                   # already good  
    "projection head architecture non-linear MLP",           # rephrased
]

def get_infoed_query(query):
    return query + hyde_query(query)

def rephrase_queries(queries):
    new_q = []
    for query in queries:
        new_q.append(get_infoed_query(query))
    return new_q

def scoreboard(distances):
    score = 0
    weights = [1.0, 0.75, 0.55, 0,35]
    for val, weights in zip(distances[0], weights):
        if val > 1.35:
            score += -1 * weights
        elif val < 1:
            score += 1 * weights
        else:
            score += 0.5 * weights
    return round(score, 2)
        

## Firstly let's get better queries using hyde
queries_new = rephrase_queries(queries)

for query in queries_new:    
    results = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query[:40]}...")
    print(f"Score: {scoreboard(results['distances'])}")

print("="*20)

for query in queries:    
    results = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query[:40]}...")
    print(f"Score: {scoreboard(results['distances'])}")

NameError: name 'collection' is not defined

In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent))
import os
from pipeline.document_processor import PreprocessDocument

obj = PreprocessDocument(os.path.join(os.getcwd(), "../docs")).save_to_chroma()
response = obj.query_kb("projection head architecture non-linear MLP")


/Users/ayushjaswal/Developer/personal-projects/groundedQAproject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading 2 files with 9 workers


2it [00:09,  4.98s/it]


Stored 106 chunks from SimCLR.pdf
Stored 78 chunks from CLRMR.pdf
Total in collection: 184


In [4]:
response = obj.query_kb("What is Contrastive learning for Music?")
response

{'ids': [['CLRMR_8', 'CLRMR_14', 'CLRMR_41', 'CLRMR_2']],
 'embeddings': None,
 'documents': [['In this paper, we combine the insights of a simple contrastive learning framework for images, SimCLR [17], with recent advances in representation learning for audio in the time domain [26]. We also contribute a pipeline of data augmentations on musical audio, to form a simple framework for self-supervised, contrastive learning of representations of raw waveforms of music. To compare the effectiveness of this simple framework compared to a more complex self-supervised learning objective, we also evaluate representations learned by contrastive predictive coding (CPC) [15]. The self-supervised models are evaluated on the downstream music tagging task, enabling us to evaluate their versatility: music tags describe many characteristics of music, e.g., genre, instrumentation and dynamics. Our key contributions are the following.',
   'This work builds on SimCLR, a simple contrastive learning frame

In [9]:
# queries = [
#     "what is SimCLR?",
#     "what data augmentation does SimCLR use?",
#     "how does the contrastive loss work?",
#     "what is the projection head?",
# ]

queries = [
    "What is Contrastive learning for Music?",        # rephrased
    "what data augmentation does SimCLR use?",               # already good
    "how does the Proxy loss for Contrsative Learning in Music work?",                   # already good  
    "projection head architecture non-linear MLP",           # rephrased
]

def get_infoed_query(query):
    return query + hyde_query(query)

def rephrase_queries(queries):
    new_q = []
    for query in queries:
        new_q.append(get_infoed_query(query))
    return new_q

def scoreboard(distances):
    score = 0
    weights = [1.0, 0.75, 0.55, 0,35]
    for val, weights in zip(distances[0], weights):
        if val > 1.35:
            score += -1 * weights
        elif val < 1:
            score += 1 * weights
        else:
            score += 0.5 * weights
    return round(score, 2)
        

## Firstly let's get better queries using hyde
queries_new = rephrase_queries(queries)

for query in queries_new:    
    results_new = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query[:40]}...")
    print(f"Score: {scoreboard(results_new['distances'])}")

print("="*20)

for query in queries:    
    results = collection.query(
        query_texts=[query],
        n_results=4,
        include=["documents", "distances"]
    )
    print(f"Query: {query[:40]}...")
    print(f"Score: {scoreboard(results['distances'])}")

Query: What is Contrastive learning for Music?C...
Score: 0
Query: what data augmentation does SimCLR use?S...
Score: 0
Query: how does the Proxy loss for Contrsative ...
Score: 0
Query: projection head architecture non-linear ...
Score: 0
Query: What is Contrastive learning for Music?...
Score: 0
Query: what data augmentation does SimCLR use?...
Score: 0
Query: how does the Proxy loss for Contrsative ...
Score: 0
Query: projection head architecture non-linear ...
Score: 0


In [10]:
queries_new[2]

'how does the Proxy loss for Contrsative Learning in Music work?Proxy Loss for Contrastive Learning in Music operates on the principle of maximizing the agreement between augmented views of the same music data, while simultaneously minimizing the agreement between views of different examples. This technique leverages a proxy discriminator network that is trained in tandem with the music encoder. The proxy loss function incorporates the proxy outputs of the network, typically evaluated using cross-entropy or pairwise ranking losses.\n\nIn the proxy loss framework, the encoder and discriminator are first initialized with a random assignment of representations. During training, the proxy discriminator evaluates the agreement between pair-wise outputs generated from different augmentations of the same data, and the representations from different examples. The contrastive loss encourages the similarity of augmented views belonging to the same class and the dissimilarity between views from'